# AYA Edge 0.2 — Dual Tesla T4 DDP infrastructure smoke test

This notebook validates one distributed forward/backward/optimizer step on exactly two Tesla T4 GPUs. It does **not** start production pretraining, certify model quality, or validate agent competence. Synthetic token IDs are used because the production corpus and final tokenizer artifact are not approved.


## Safety gates

- Clone and verify a pinned AYA Edge commit.
- Require exactly two visible Tesla T4 GPUs; never silently fall back to one GPU.
- Run repository tests before CUDA work.
- Use synthetic IDs from the 268-token development adapter only for infrastructure.
- Keep production training blocked while model/tokenizer/corpus approval is incomplete.


In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'Aya-Edge-0.1'
REPO_URL = 'https://github.com/hygienhub03-dev/Aya-Edge-0.1.git'
BRANCH = 'feature/aya-edge-dual-t4-smoke'
PINNED_COMMIT = 'PENDING_DUAL_T4_BASE_COMMIT'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PINNED_COMMIT, (actual_commit, PINNED_COMMIT)
print('Repository:', REPO)
print('Pinned commit:', actual_commit)
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Free disk GiB:', round(shutil.disk_usage(WORK).free / 2**30, 2))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)


In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU count:', torch.cuda.device_count())
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select exactly two Kaggle Tesla T4 GPUs; single-GPU fallback is forbidden.')
gpu_report = []
for index in range(2):
    props = torch.cuda.get_device_properties(index)
    row = {'index': index, 'name': props.name, 'memory_gib': round(props.total_memory / 2**30, 2), 'compute_capability': f'{props.major}.{props.minor}'}
    gpu_report.append(row)
    print(row)
if any('T4' not in row['name'].upper() for row in gpu_report):
    raise RuntimeError(f'Expected Tesla T4 devices, found {gpu_report}')


In [ ]:
test = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, capture_output=True)
print(test.stdout)
if test.stderr:
    print(test.stderr[-12000:])
if test.returncode != 0:
    raise RuntimeError('Repository tests failed; stop before distributed execution.')
print('Repository tests: PASSED')


In [ ]:
import yaml
from aya_edge import AyaEdgeConfig, AyaEdgeForCausalLM, count_active_parameters, count_parameters
from aya_edge.tokenizer import ByteLevelTokenizer

raw = yaml.safe_load(Path('configs/aya_edge_0_2.yaml').read_text(encoding='utf-8'))
cfg = AyaEdgeConfig(**raw['model'])
development_tokenizer = ByteLevelTokenizer()
model = AyaEdgeForCausalLM(cfg)
print('Model vocabulary:', cfg.vocab_size)
print('Development tokenizer vocabulary:', development_tokenizer.vocab_size)
print('Total trainable parameters:', f'{count_parameters(model):,}')
print('Structural active estimate:', f'{count_active_parameters(model):,}')
assert cfg.vocab_size == 32768
assert development_tokenizer.vocab_size == 268
assert cfg.vocab_size != development_tokenizer.vocab_size
PRODUCTION_TRAINING_ALLOWED = False
assert not PRODUCTION_TRAINING_ALLOWED
print('PRODUCTION TRAINING BLOCKED: final BPE artifact and approved corpus are pending.')
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
worker = Path('scripts/dual_t4_ddp_smoke.py')
assert worker.exists()
print('DDP worker:', worker)
launch = subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', str(worker)], text=True, capture_output=True, timeout=1800)
print(launch.stdout)
if launch.stderr:
    print(launch.stderr[-12000:])
if launch.returncode != 0:
    raise RuntimeError(f'Dual-GPU DDP smoke test failed with exit code {launch.returncode}.')
print('Dual-T4 DDP smoke step: PASSED')


In [ ]:
metrics_path = Path('outputs/dual_t4_smoke/metrics.json')
checkpoint_path = Path('outputs/dual_t4_smoke/aya_edge_02_ddp_smoke.pt')
assert metrics_path.exists(), 'Metrics file missing'
assert checkpoint_path.exists(), 'Checkpoint missing'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2))
assert metrics['status'] == 'PASSED'
assert metrics['world_size'] == 2
assert metrics['parameters'] == 310115904
assert metrics['embedding_weights_changed'] is True
assert metrics['production_training_allowed'] is False
payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert payload['step'] == 1 and payload['model']
assert payload['model_vocab_size'] == 32768
assert payload['tokenizer_vocab_size'] == 268
print('Checkpoint integrity: PASSED')


## Final status

Download the metrics and checkpoint from Kaggle's Output panel if needed. The checkpoint is an infrastructure smoke-test artifact only. It must not initialize production training.
